# 02 - Train Crop Identifier (EfficientNet-B0)

Stage 2: Train the crop identification model (EfficientNet-B0).

Reads dataset_split/{train,val}/<crop>/ produced by 01_prepare_crop_dataset.py.
Uses a WeightedRandomSampler to counter the crop-level class imbalance
(Cotton has ~3.5x fewer images than Pepper Bell).

Install deps:
    pip install torch torchvision timm albumentations scikit-learn --break-system-packages

## Imports & Configuration

In [1]:
import torch
print(torch.cuda.is_available()) # Should be True
print(torch.version.cuda)       # Shows the CUDA version PyTorch expects


True
12.6


In [2]:
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from sklearn.metrics import classification_report, confusion_matrix
import cv2

DATA_DIR = Path(r"Z:\Projects\Smart-Farming\Datasets\dataset_split")
MODEL_OUT = Path(r"Z:\Projects\Smart-Farming\models\crop_identifier_v1.pth")
LABELS_OUT = Path(r"Z:\Projects\Smart-Farming\models\crop_identifier_labels.json")
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LR = 3e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## `AlbumentationsImageFolder`

Wraps ImageFolder to apply Albumentations transforms.

In [3]:
class AlbumentationsImageFolder(Dataset):
    """Wraps ImageFolder to apply Albumentations transforms."""
    def __init__(self, root, transform):
        self.base = ImageFolder(root)
        self.transform = transform
        self.classes = self.base.classes

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        path, label = self.base.samples[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, label

## `build_weighted_sampler`

In [4]:
def build_weighted_sampler(dataset):
    targets = [label for _, label in dataset.base.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

## `train`

In [5]:
from tqdm.auto import tqdm  # auto automatically switches to a clean notebook layout

def train():
    train_ds = AlbumentationsImageFolder(DATA_DIR / "train", train_tf)
    val_ds = AlbumentationsImageFolder(DATA_DIR / "val", val_tf)
    num_classes = len(train_ds.classes)
    print(f"Classes: {train_ds.classes}")

    sampler = build_weighted_sampler(train_ds)
    
    # NOTE: If it still freezes completely, remember to change num_workers to 0!
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    print("Downloading/Loading model weights...")
    model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=num_classes)
    model.to(DEVICE)
    print(f"Model successfully loaded on {DEVICE}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience, patience_counter = 5, 0

    for epoch in range(EPOCHS):
        print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
        
        # 1. TRAINING LOOP WITH LIVE PROGRESS
        model.train()
        running_loss = 0.0
        
        # Wrap train_loader in tqdm for a live update per batch
        train_pbar = tqdm(train_loader, desc="  Training", leave=False)
        for images, labels in train_pbar:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            
            # This updates the progress bar text in real-time
            current_batch_loss = loss.item()
            train_pbar.set_postfix({"batch_loss": f"{current_batch_loss:.4f}"})
            
        scheduler.step()
        train_loss = running_loss / len(train_ds)

        # 2. VALIDATION LOOP WITH LIVE PROGRESS
        model.eval()
        correct, total = 0, 0
        
        # Wrap val_loader in tqdm
        val_pbar = tqdm(val_loader, desc="  Validating", leave=False)
        with torch.no_grad():
            for images, labels in val_pbar:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        val_acc = correct / total

        # Print final epoch metrics
        print(f"Result -> train_loss: {train_loss:.4f} - val_acc: {val_acc:.4f}")

        # Early Stopping & Saving Logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), MODEL_OUT)
            with open(LABELS_OUT, "w") as f:
                json.dump(train_ds.classes, f)
            print(f"  -> 🎉 Saved new best model! (val_acc={val_acc:.4f})")
        else:
            patience_counter += 1
            print(f"  -> No improvement. Early stopping counter: {patience_counter}/{patience}")
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    print(f"\nFinished! Best val_acc: {best_val_acc:.4f}. Model saved to {MODEL_OUT}")


## Run

In [6]:
train()

Classes: ['Cotton', 'Groundnut', 'Pepper Bell', 'Potato', 'Tomato']
Downloading/Loading model weights...


Model successfully loaded on cuda

--- Epoch 1/20 ---


Result -> train_loss: 0.2026 - val_acc: 0.9840
  -> 🎉 Saved new best model! (val_acc=0.9840)

--- Epoch 2/20 ---


Result -> train_loss: 0.0788 - val_acc: 0.9869
  -> 🎉 Saved new best model! (val_acc=0.9869)

--- Epoch 3/20 ---


Result -> train_loss: 0.0483 - val_acc: 0.9880
  -> 🎉 Saved new best model! (val_acc=0.9880)

--- Epoch 4/20 ---


...

Result -> train_loss: 0.0035 - val_acc: 0.9944
  -> No improvement. Early stopping counter: 1/5

--- Epoch 17/20 ---


Result -> train_loss: 0.0022 - val_acc: 0.9960
  -> 🎉 Saved new best model! (val_acc=0.9960)

--- Epoch 18/20 ---


Result -> train_loss: 0.0017 - val_acc: 0.9962
  -> 🎉 Saved new best model! (val_acc=0.9962)

--- Epoch 19/20 ---


Result -> train_loss: 0.0011 - val_acc: 0.9955
  -> No improvement. Early stopping counter: 1/5

--- Epoch 20/20 ---


Result -> train_loss: 0.0029 - val_acc: 0.9960
  -> No improvement. Early stopping counter: 2/5

Finished! Best val_acc: 0.9962. Model saved to Z:\Projects\Smart-Farming\models\crop_identifier_v1.pth
